# 드론 예측정비 결과 검증 노트북

`drone_pm.py`가 만든 `outputs/` 결과가 맞는지 **단계별로 직접 계산해 보며** 확인한다.

- 결과 파일은 읽기만 하고, 다시 계산할 때는 임시 폴더를 쓴다.
- 각 단계 끝의 `check()`가 통과 / 실패를 기록하고, 마지막에 한꺼번에 정리한다.
- 같은 내용을 한 번에 돌리는 스크립트: `reports/verify_results.py`

| 단계 | 확인 내용 |
|---|---|
| 1 | 정답(`failure_within_30days`) 정의 |
| 2 | 제출 파일 형식 |
| 3 | 기간 분할 |
| 4 | 미래 정보 누수 |
| 5 | 데이터 처리(999 · 비용) |
| 6 | 검증 성능 재현 · 기준선 |
| 7 | 위험 표시 기준 |
| 8 | 최종 예측 · 재점검 20대 |
| 9 | 보고서 숫자 |

In [1]:
import sys, tempfile, re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

ROOT = Path.cwd().parent if Path.cwd().name == "reports" else Path.cwd()
sys.path.insert(0, str(ROOT))
import drone_pm as pm

DATA, OUT = ROOT / "data", ROOT / "outputs"
TARGET = "failure_within_30days"
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)

results = []
def check(name, ok, detail=""):
    results.append({"항목": name, "결과": "통과" if ok else "실패", "내용": detail})
    print(f"[{'통과' if ok else '실패'}] {name}" + (f" — {detail}" if detail else ""))

read = lambda p, **kw: pd.read_csv(p, encoding="utf-8-sig", **kw)
train = read(DATA / "drone_train.csv", parse_dates=["maintenance_date"])
test = read(DATA / "drone_test.csv", parse_dates=["maintenance_date"])
month = read(DATA / "drone_this_month.csv", parse_dates=["maintenance_date"])
print(f"학습 {len(train)}행 · 시험 {len(test)}행 · 이번 달 {len(month)}행")

학습 2346행 · 시험 1241행 · 이번 달 106행


## 1. 정답 정의

> 정비 뒤 30일 안에 사후정비 · 긴급정비가 있었으면 1, 없으면 0

기록만 보고 정답을 직접 만들어 주어진 정답과 비교한다. 먼저 DR-0001의 처음 몇 줄로 눈으로 확인한다.

In [2]:
al = pd.concat([train, test]).sort_values(["equipment_id", "maintenance_date"])
fails = al[al.maintenance_type.isin(["사후정비", "긴급정비"])]
fail_dates = fails.groupby("equipment_id").maintenance_date.apply(np.array)

def next_failure_gap(eid, d):
    fd = fail_dates.get(eid, np.array([], dtype="datetime64[ns]"))
    gap = (fd - np.datetime64(d)) / np.timedelta64(1, "D")
    gap = gap[gap > 0]
    return gap.min() if len(gap) else np.nan

ex = train[train.equipment_id == "DR-0001"].head(6).copy()
ex["다음 고장까지(일)"] = [next_failure_gap(r.equipment_id, r.maintenance_date) for r in ex.itertuples()]
ex["다시 만든 정답"] = (ex["다음 고장까지(일)"] <= 30).astype(int)
ex[["maintenance_date", "maintenance_type", "다음 고장까지(일)", "다시 만든 정답", TARGET]]

,maintenance_date,maintenance_type,다음 고장까지(일),다시 만든 정답,failure_within_30days
0,2023-02-27,예방정비,18.0,1,1
1,2023-03-17,긴급정비,317.0,0,0
2,2023-04-22,예방정비,281.0,0,0
3,2023-07-02,예방정비,210.0,0,0
4,2023-08-25,예방정비,156.0,0,0
5,2023-10-20,예방정비,100.0,0,0


In [3]:
gaps = np.array([next_failure_gap(r.equipment_id, r.maintenance_date) for r in train.itertuples()])
derived = (gaps <= 30).astype(int)
agree = (derived == train[TARGET].values).mean()
check("다시 만든 정답이 주어진 정답과 일치", agree == 1.0, f"일치율 {agree:.4f} ({len(train)}행)")
print("※ 시험 데이터 정답을 만드는 데는 쓰지 않는다 (채점용 정답은 강사만 가짐)")

[통과] 다시 만든 정답이 주어진 정답과 일치 — 일치율 1.0000 (2346행)
※ 시험 데이터 정답을 만드는 데는 쓰지 않는다 (채점용 정답은 강사만 가짐)


## 2. 제출 파일 형식 (`test_predictions_2025.csv`)

In [4]:
sub = read(OUT / "test_predictions_2025.csv", dtype=str)
raw_test = read(DATA / "drone_test.csv", dtype=str)
check("열 이름", list(sub.columns) == ["equipment_id", "maintenance_date", "prob"], str(list(sub.columns)))
check("행 수 1,241", len(sub) == len(raw_test) == 1241, f"{len(sub)}행")
check("드론 · 날짜가 시험 데이터와 순서까지 일치",
      (sub[["equipment_id", "maintenance_date"]].values == raw_test[["equipment_id", "maintenance_date"]].values).all())
check("날짜 형식 YYYY-MM-DD", sub.maintenance_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all())
p = pd.to_numeric(sub.prob, errors="coerce")
check("확률 0~1, 빈 값 없음", p.notna().all() and p.between(0, 1).all(), f"{p.min():.4f} ~ {p.max():.4f}")
sub.head()

[통과] 열 이름 — ['equipment_id', 'maintenance_date', 'prob']
[통과] 행 수 1,241 — 1241행
[통과] 드론 · 날짜가 시험 데이터와 순서까지 일치
[통과] 날짜 형식 YYYY-MM-DD
[통과] 확률 0~1, 빈 값 없음 — 0.0125 ~ 0.9944


,equipment_id,maintenance_date,prob
0,DR-0001,2025-01-04,0.095081
1,DR-0001,2025-03-06,0.044393
2,DR-0001,2025-05-17,0.047639
3,DR-0001,2025-07-28,0.063028
4,DR-0001,2025-08-09,0.31467


## 3. 기간 분할

무작위가 아니라 기간으로 나눴는지, 학습 행의 정답 창(30일)이 검증 기간에 닿지 않는지 본다.

In [5]:
purge_end = pm.VALID_START - pd.Timedelta(days=pm.PURGE_DAYS)
fit = train[train.maintenance_date < purge_end]
gapzone = train[(train.maintenance_date >= purge_end) & (train.maintenance_date < pm.VALID_START)]
val = train[train.maintenance_date >= pm.VALID_START]
split = pd.DataFrame({
    "구간": ["학습", "제외(경계)", "검증", "시험"],
    "시작": [fit.maintenance_date.min(), gapzone.maintenance_date.min(), val.maintenance_date.min(), test.maintenance_date.min()],
    "끝": [fit.maintenance_date.max(), gapzone.maintenance_date.max(), val.maintenance_date.max(), test.maintenance_date.max()],
    "행 수": [len(fit), len(gapzone), len(val), len(test)],
})
display(split)
last_window = fit.maintenance_date.max() + pd.Timedelta(days=30)
check("학습 마지막 날 + 30일 < 검증 시작", last_window < pm.VALID_START, f"{last_window:%Y-%m-%d} < {pm.VALID_START:%Y-%m-%d}")
check("학습이 모두 검증보다 앞 (무작위 분할 아님)", fit.maintenance_date.max() < val.maintenance_date.min())
check("학습 · 검증에 2025 데이터 없음", train.maintenance_date.max() < pd.Timestamp("2025-01-01"))

,구간,시작,끝,행 수
0,학습,2023-01-01,2024-05-31,1611
1,제외(경계),2024-06-01,2024-06-29,92
2,검증,2024-07-01,2024-12-31,643
3,시험,2025-01-01,2025-12-31,1241


[통과] 학습 마지막 날 + 30일 < 검증 시작 — 2024-06-30 < 2024-07-01
[통과] 학습이 모두 검증보다 앞 (무작위 분할 아님)
[통과] 학습 · 검증에 2025 데이터 없음


## 4. 미래 정보 누수

**방법**: 어떤 정비 기록을 고른 뒤, 그 날짜 **뒤의 기록을 전부 지운** 데이터로 피처를 다시 만든다.
미래 기록을 쓰지 않았다면 원래 피처와 값이 똑같아야 한다.

In [6]:
tmp = tempfile.TemporaryDirectory()
pm.OUT = Path(tmp.name)                      # clean()의 로그가 outputs/를 덮어쓰지 않게 함
raw = pm.load()
full = pm.build_features(pm.clean(raw)).set_index("row_id")

HIST = ["fail_cnt_30d", "fail_cnt_90d", "fail_cnt_180d", "fail_cnt_365d", "maint_cnt_90d",
        "same_cat_fail_365d", "unsuccess_cnt_180d", "severity_sum_180d", "days_since_prev_failure",
        "days_since_last_pm", "hours_since_last_pm", "n_prior_records", "hours_per_day_recent",
        "prev_maintenance_type", "days_since_last_maintenance"]

def features_without_future(rid):
    d = raw.loc[raw.row_id == rid, "maintenance_date"].iloc[0]
    cut = raw[(raw.maintenance_date < d) | ((raw.maintenance_date == d) & (raw.row_id <= rid))]
    return pm.build_features(pm.clean(cut)).set_index("row_id").loc[rid, HIST]

# 예시 1건: DR-0115의 2025년 기록 하나
rid = raw[(raw.equipment_id == "DR-0115") & (raw.split == "test")].row_id.iloc[10]
cmp = pd.DataFrame({"전체 데이터로 계산": full.loc[rid, HIST], "미래 기록 지우고 계산": features_without_future(rid)})
print(raw.loc[raw.row_id == rid, ["equipment_id", "maintenance_date", "split"]].to_string(index=False))
cmp


[2] 이상치 수정 9건 -> outputs/cleaning_log.csv (삭제한 행 0건)



[2] 이상치 수정 9건 -> outputs/cleaning_log.csv (삭제한 행 0건)


equipment_id maintenance_date split
     DR-0115       2025-02-23  test


,전체 데이터로 계산,미래 기록 지우고 계산
fail_cnt_30d,6.0,6.0
fail_cnt_90d,19.0,19.0
fail_cnt_180d,36.0,36.0
fail_cnt_365d,55.0,55.0
maint_cnt_90d,20.0,20.0
same_cat_fail_365d,19.0,19.0
unsuccess_cnt_180d,11.0,11.0
severity_sum_180d,125.0,125.0
days_since_prev_failure,3.0,3.0
days_since_last_pm,25.0,25.0


In [7]:
sample = raw[raw.maintenance_date >= "2024-07-01"].sample(20, random_state=0).row_id
bad = []
for rid in sample:
    f, g = features_without_future(rid), full.loc[rid, HIST]
    for c in HIST:
        same = f[c] == g[c] if isinstance(f[c], str) else np.isclose(float(f[c]), float(g[c]), equal_nan=True)
        if not same:
            bad.append((rid, c, f[c], g[c]))
check("이력 피처가 미래 기록 없이도 똑같음", not bad, f"표본 {len(sample)}건 × {len(HIST)}개, 불일치 {len(bad)}개")
check("모델 입력에 availability_rate · 정답 없음", "availability_rate" not in pm.FEATURES and TARGET not in pm.FEATURES)


[2] 이상치 수정 9건 -> outputs/cleaning_log.csv (삭제한 행 0건)



[2] 이상치 수정 9건 -> outputs/cleaning_log.csv (삭제한 행 0건)



[2] 이상치 수정 9건 -> outputs/cleaning_log.csv (삭제한 행 0건)



[2] 이상치 수정 8건 -> outputs/cleaning_log.csv (삭제한 행 0건)



[2] 이상치 수정 9건 -> outputs/cleaning_log.csv (삭제한 행 0건)



[2] 이상치 수정 9건 -> outputs/cleaning_log.csv (삭제한 행 0건)


... (11 outputs left out)


[2] 이상치 수정 8건 -> outputs/cleaning_log.csv (삭제한 행 0건)



[2] 이상치 수정 9건 -> outputs/cleaning_log.csv (삭제한 행 0건)



[2] 이상치 수정 9건 -> outputs/cleaning_log.csv (삭제한 행 0건)


[통과] 이력 피처가 미래 기록 없이도 똑같음 — 표본 20건 × 15개, 불일치 0개
[통과] 모델 입력에 availability_rate · 정답 없음


## 5. 데이터 처리

- `downtime_hours = 999` → 결측 코드. 값을 지어내지 않고 **모름(빈 값)** 으로 둔다.
- `maintenance_cost` **1만 이상이면서 100의 배수** → 100배 단위 오입력으로 보고 ÷100.
- 직전 정비 없음(드론별 첫 기록) → **비워 두고** `is_first_maintenance = 1`. 0('방금 정비함') · 중앙값('51일 전 정비')으로 채우지 않는다.

In [8]:
r = raw.set_index("row_id")
is999 = r.downtime_hours >= 999
check("999는 모두 빈 값 + downtime_missing=1", full.loc[is999, "downtime_hours"].isna().all()
      and (full.loc[is999, "downtime_missing"] == 1).all(), f"{int(is999.sum())}건")
check("999가 아닌 운용중단 시간은 그대로", (full.loc[~is999, "downtime_hours"] == r.loc[~is999, "downtime_hours"]).all())

# 직전 정비 없음 -> 비워 두고 표시 (0 · 중앙값으로 채우지 않음)
first = full.is_first_maintenance == 1
display(full.loc[first, ["equipment_id", "maintenance_date", "split", "days_since_last_maintenance",
                         "hours_per_day_recent", "is_first_maintenance"]].head(3))
check("직전 정비 없는 기록은 빈 값 + is_first_maintenance=1",
      full.loc[first, ["days_since_last_maintenance", "hours_per_day_recent"]].isna().all().all()
      and (r.days_since_last_maintenance.isna() == first).all(),
      f"{int(first.sum())}건, 분할별 {full[first].split.value_counts().to_dict()}")
check("0으로 채운 값 없음 (경과일 최솟값 > 0)", full.days_since_last_maintenance.min() > 0)

rule = (r.maintenance_cost >= 10000) & (r.maintenance_cost % 100 == 0)
normal = r[~rule]
typ = normal.groupby("maintenance_type").maintenance_cost.agg(["min", "median", "max"])
display(typ)
fixed = pd.DataFrame({"원래": r.loc[rule, "maintenance_cost"], "처리 후": full.loc[rule, "maintenance_cost"],
                      "정비 구분": r.loc[rule, "maintenance_type"]})
display(fixed)
check("÷100 대상 = 규칙에 맞는 행", set(full.index[full.cost_fixed == 1]) == set(rule[rule].index), f"{int(rule.sum())}건")
check("처리 후 값이 같은 정비 구분 정상 범위 안",
      all(v <= typ.loc[t, "max"] for v, t in zip(fixed["처리 후"], fixed["정비 구분"])))
check("정상 최댓값과 오입력 최솟값 사이에 값 없음",
      r.maintenance_cost.between(normal.maintenance_cost.max() + 1, r.loc[rule, "maintenance_cost"].min() - 1).sum() == 0,
      f"{normal.maintenance_cost.max()} ~ {r.loc[rule, 'maintenance_cost'].min()}")

[통과] 999는 모두 빈 값 + downtime_missing=1 — 4건
[통과] 999가 아닌 운용중단 시간은 그대로


,equipment_id,maintenance_date,split,days_since_last_maintenance,hours_per_day_recent,is_first_maintenance
row_id,,,,,,
0,DR-0001,2023-02-27,train,NaN,NaN,1
14,DR-0002,2023-01-22,train,NaN,NaN,1
29,DR-0003,2023-03-05,train,NaN,NaN,1


[통과] 직전 정비 없는 기록은 빈 값 + is_first_maintenance=1 — 150건, 분할별 {'train': 150}
[통과] 0으로 채운 값 없음 (경과일 최솟값 > 0)


,min,median,max
maintenance_type,,,
긴급정비,40,247.5,611
사후정비,30,109.0,449
예방정비,8,19.0,42


,원래,처리 후,정비 구분
row_id,,,
716,15000,150,사후정비
837,16800,168,사후정비
1038,16200,162,사후정비
1408,18300,183,사후정비
1814,19000,190,사후정비


[통과] ÷100 대상 = 규칙에 맞는 행 — 5건
[통과] 처리 후 값이 같은 정비 구분 정상 범위 안
[통과] 정상 최댓값과 오입력 최솟값 사이에 값 없음 — 611 ~ 15000


## 6. 검증 성능 재현 · 기준선

같은 기간 분할로 모델을 다시 학습해 `validation_metrics.csv`와 비교한다.
월 20대 적중은 **매달 드론별 최근 기록**을 확률 순으로 20대 고른 뒤 실제 재고장 수를 센다 (여기서 직접 계산).

In [9]:
metrics = read(OUT / "validation_metrics.csv").set_index("model")
df = full.reset_index()
tr = df[df.split == "train"]
fit_f = tr[tr.maintenance_date < purge_end]
val_f = tr[tr.maintenance_date >= pm.VALID_START]

def monthly_hits(d, score, k=20):
    d = d.assign(score=score, ym=d.maintenance_date.dt.to_period("M"))
    d = d.sort_values(["maintenance_date", "row_id"]).groupby(["ym", "equipment_id"]).tail(1)
    return {str(m): int(g.nlargest(k, "score")[TARGET].sum()) for m, g in d.groupby("ym")}, d

rows, preds = [], {}
for name, model in pm.make_models().items():
    model.fit(fit_f[pm.FEATURES], fit_f[TARGET])
    p = model.predict_proba(val_f[pm.FEATURES])[:, 1]
    preds[name] = p
    rows.append({"모델": name, "PR-AUC 재계산": average_precision_score(val_f[TARGET], p),
                 "PR-AUC 파일": metrics.at[name, "PR_AUC"],
                 "적중 재계산": sum(monthly_hits(val_f, p)[0].values()), "적중 파일": metrics.at[name, "top20_hits_total"]})
rep = pd.DataFrame(rows); display(rep.round(4))
for x in rows:
    check(f"{x['모델']} 재현", abs(x["PR-AUC 재계산"] - x["PR-AUC 파일"]) < 1e-3 and x["적중 재계산"] == x["적중 파일"])

,모델,PR-AUC 재계산,PR-AUC 파일,적중 재계산,적중 파일
0,로지스틱회귀,0.7772,0.7772,35,35.0
1,랜덤포레스트,0.7989,0.7989,39,39.0
2,그래디언트부스팅,0.7879,0.7879,38,38.0


[통과] 로지스틱회귀 재현
[통과] 랜덤포레스트 재현
[통과] 그래디언트부스팅 재현


In [10]:
_, latest = monthly_hits(val_f, np.zeros(len(val_f)))
exp_rand = sum(min(20, len(g)) * g[TARGET].mean() for _, g in latest.groupby("ym"))
file_rand = metrics.loc[metrics.index.str.contains("무작위"), "top20_hits_total"].iloc[0]
check("무작위 기준선 적중 ≈ 식으로 구한 기대값 (매달 20 × 재고장 비율)", abs(exp_rand - file_rand) < 0.5,
      f"기대값 {exp_rand:.2f} / 파일 {file_rand}")
age_pr = average_precision_score(val_f[TARGET], val_f.equipment_age_years)
check("경과연수순 PR-AUC 재현", abs(age_pr - metrics.loc[metrics.index.str.contains("경과연수"), "PR_AUC"].iloc[0]) < 1e-3, f"{age_pr:.4f}")

best = metrics[~metrics.index.str.startswith("기준선")].PR_AUC.idxmax()
repeat = pm.repeat_failure_drones(tr)
keep = ~val_f.equipment_id.isin(repeat).values
pr_ex = average_precision_score(val_f[TARGET].values[keep], preds[best][keep])
check(f"반복 고장 드론 {repeat} 제외 PR-AUC 재현", abs(pr_ex - metrics.at[best, "PR_AUC_excl_repeat"]) < 1e-3,
      f"{pr_ex:.4f} (전체 {metrics.at[best, 'PR_AUC']:.4f})")
check("반복 고장 드론을 빼도 기준선보다 높음",
      pr_ex > metrics.loc[metrics.index.str.startswith("기준선"), "PR_AUC_excl_repeat"].max())
metrics

[통과] 무작위 기준선 적중 ≈ 식으로 구한 기대값 (매달 20 × 재고장 비율) — 기대값 13.65 / 파일 13.56
[통과] 경과연수순 PR-AUC 재현 — 0.3479
[통과] 반복 고장 드론 ['DR-0034', 'DR-0115'] 제외 PR-AUC 재현 — 0.6124 (전체 0.7989)
[통과] 반복 고장 드론을 빼도 기준선보다 높음


,PR_AUC,ROC_AUC,top20_hits_total,positives_total,PR_AUC_excl_repeat
model,,,,,
기준선_무작위(1000회 평균),0.2803,0.4996,13.56,176,0.2156
기준선_경과연수순,0.3479,0.6215,18.34,176,0.2642
로지스틱회귀,0.7772,0.8734,35.00,176,0.5681
랜덤포레스트,0.7989,0.8860,39.00,176,0.6124
그래디언트부스팅,0.7879,0.8835,38.00,176,0.5985


## 7. 위험 표시 기준

**비용 = 놓친 재고장 수 × 놓침 비용 + 기준 이상 건수 × 60만원**.
검증 기간 예측만으로 0.01~0.99를 모두 계산해 가장 싼 기준을 찾는다 (2025 시험 데이터는 쓰지 않음).

In [11]:
y, pv = val_f[TARGET].values, preds[best]
ts = np.round(np.arange(0.01, 1.0, 0.01), 2)
curve = pd.DataFrame({"기준": ts,
                      "표시": [int((pv >= t).sum()) for t in ts],
                      "놓침": [int(((pv < t) & (y == 1)).sum()) for t in ts]})
for miss in (300, 600):
    curve[f"비용_{miss}"] = curve.놓침 * miss + curve.표시 * 60
costs = read(OUT / "threshold_costs.csv")
chosen = {}
for miss in (300, 600):
    mine = curve.sort_values([f"비용_{miss}", "기준"], ascending=[True, False]).iloc[0]
    theirs = costs.sort_values([f"cost_{miss}", "threshold"], ascending=[True, False]).iloc[0]
    chosen[miss] = mine.기준
    check(f"놓침 {miss}만원 기준 재현", mine.기준 == theirs.threshold and mine[f"비용_{miss}"] == theirs[f"cost_{miss}"],
          f"{mine.기준:.2f} / {int(mine[f'비용_{miss}']):,}만원")
check("놓침 비용이 커지면 기준이 같거나 낮아짐", chosen[600] <= chosen[300], f"{chosen[300]:.2f} → {chosen[600]:.2f}")
curve[(curve.기준 >= 0.10) & (curve.기준 <= 0.26)]

[통과] 놓침 300만원 기준 재현 — 0.22 / 23,340만원
[통과] 놓침 600만원 기준 재현 — 0.11 / 29,640만원
[통과] 놓침 비용이 커지면 기준이 같거나 낮아짐 — 0.22 → 0.11


,기준,표시,놓침,비용_300,비용_600
9,0.10,379,12,26340,29940
10,0.11,354,14,25440,29640
11,0.12,342,16,25320,30120
12,0.13,333,17,25080,30180
13,0.14,318,19,24780,30480
14,0.15,311,21,24960,31260
15,0.16,302,23,25020,31920
16,0.17,290,24,24600,31800
17,0.18,283,25,24480,31980
18,0.19,272,25,23820,31320


## 8. 최종 예측 · 2026-01 재점검 20대

학습 데이터 전체(2023-01~2024-12)로 다시 학습해 제출 파일과 재점검 계획을 재현한다.

In [12]:
te, mo = df[df.split == "test"], df[df.split == "this_month"]
final = pm.make_models()[best]
final.fit(tr[pm.FEATURES], tr[TARGET])
p_test = final.predict_proba(te[pm.FEATURES])[:, 1]
diff = np.abs(p_test - read(OUT / "test_predictions_2025.csv").prob.values).max()
check("시험 예측 재현 (소수 6자리 반올림 오차 이내)", diff < 1e-5, f"최대 차이 {diff:.1e}")

latest_m = mo.assign(prob=final.predict_proba(mo[pm.FEATURES])[:, 1]) \
             .sort_values(["maintenance_date", "row_id"]).groupby("equipment_id").tail(1) \
             .sort_values("prob", ascending=False)
plan = read(OUT / "recheck_plan_2026_01.csv")
check("재점검 20대 = 드론별 최근 기록 확률 상위 20대", latest_m.equipment_id.head(20).tolist() == plan["드론"].tolist())
over = int((latest_m.prob >= chosen[300]).sum())
check("기준 이상 드론 수", over == int(read(OUT / "this_month_predictions.csv").flag_300.sum()),
      f"{over}대 / 이번 달 {len(latest_m)}대")
plan[["순위", "드론", "유형", "부대", "확률", "권고"]].head(10)

[통과] 시험 예측 재현 (소수 6자리 반올림 오차 이내) — 최대 차이 5.0e-07
[통과] 재점검 20대 = 드론별 최근 기록 확률 상위 20대
[통과] 기준 이상 드론 수 — 37대 / 이번 달 81대


,순위,드론,유형,부대,확률,권고
0,1,DR-0115,수송드론,UNIT-02,0.987,반복 고장 기체: 근본 원인 조사 · 운용 중지 검토. 모터·ESC 진동·발열 점검...
1,2,DR-0046,정찰드론,UNIT-12,0.725,"배터리 셀 전압·내부저항 측정, 충전기 점검, 필요 시 배터리팩 교체"
2,3,DR-0018,정찰드론,UNIT-10,0.704,"모터·ESC 진동·발열 점검, 베어링 상태 확인, 프로펠러 균형 점검"
3,4,DR-0038,정찰드론,UNIT-10,0.702,"IMU·GPS·짐벌 캘리브레이션, 센서 커넥터 점검"
4,5,DR-0100,수송드론,UNIT-02,0.673,"배터리 셀 전압·내부저항 측정, 충전기 점검, 필요 시 배터리팩 교체"
5,6,DR-0017,정찰드론,UNIT-06,0.633,예방정비 직후 고위험: 최근 고장 이력 계통 위주 정밀 재점검
6,7,DR-0076,감시드론,UNIT-06,0.618,"IMU·GPS·짐벌 캘리브레이션, 센서 커넥터 점검"
7,8,DR-0061,감시드론,UNIT-04,0.608,"프레임·암 균열 점검, 체결부 토크 재확인"
8,9,DR-0132,교육용드론,UNIT-11,0.480,"모터·ESC 진동·발열 점검, 베어링 상태 확인, 프로펠러 균형 점검"
9,10,DR-0113,수송드론,UNIT-11,0.462,"배터리 셀 전압·내부저항 측정, 충전기 점검, 필요 시 배터리팩 교체"


## 9. 문서 숫자 대조 (`report.md` · `recheck_plan_2026_01.md` ↔ CSV)

In [13]:
rep_md = (OUT / "report.md").read_text(encoding="utf-8")
for r in metrics.reset_index().itertuples():
    m = re.search(rf"\| {re.escape(r.model)} \| ([\d.]+) \| [\d.]+ \| ([\d.]+) \|", rep_md)
    check(f"보고서 성능표: {r.model}", bool(m) and abs(float(m.group(1)) - r.PR_AUC) < 1e-3
          and float(m.group(2)) == r.top20_hits_total)
for miss in (300, 600):
    m = re.search(rf"\| {miss}만원 \| ([\d.]+) \|", rep_md)
    check(f"보고서 위험 기준 ({miss}만원)", bool(m) and float(m.group(1)) == chosen[miss])
plan_md = (OUT / "recheck_plan_2026_01.md").read_text(encoding="utf-8")
m = re.search(r"이상 \*\*(\d+)대\*\*", plan_md)
check("계획 문서 기준 이상 드론 수", bool(m) and int(m.group(1)) == over)
check("계획 문서 20대 표가 계획 CSV와 같음", all(f"| {r['순위']} | {r['드론']} |" in plan_md for _, r in plan.iterrows()))
check("계획 문서에 모델의 한계 · 정비 담당자 판단", "## 2. 모델의 한계" in plan_md and "정비 담당자" in plan_md)

[통과] 보고서 성능표: 기준선_무작위(1000회 평균)
[통과] 보고서 성능표: 기준선_경과연수순
[통과] 보고서 성능표: 로지스틱회귀
[통과] 보고서 성능표: 랜덤포레스트
[통과] 보고서 성능표: 그래디언트부스팅
[통과] 보고서 위험 기준 (300만원)
[통과] 보고서 위험 기준 (600만원)
[통과] 계획 문서 기준 이상 드론 수
[통과] 계획 문서 20대 표가 계획 CSV와 같음
[통과] 계획 문서에 모델의 한계 · 정비 담당자 판단


## 정리

In [14]:
tmp.cleanup()
res = pd.DataFrame(results)
print(f"{(res.결과 == '통과').sum()} / {len(res)} 통과")
res

41 / 41 통과


,항목,결과,내용
0,다시 만든 정답이 주어진 정답과 일치,통과,일치율 1.0000 (2346행)
1,열 이름,통과,"['equipment_id', 'maintenance_date', 'prob']"
2,"행 수 1,241",통과,1241행
3,드론 · 날짜가 시험 데이터와 순서까지 일치,통과,
4,날짜 형식 YYYY-MM-DD,통과,
5,"확률 0~1, 빈 값 없음",통과,0.0125 ~ 0.9944
6,학습 마지막 날 + 30일 < 검증 시작,통과,2024-06-30 < 2024-07-01
7,학습이 모두 검증보다 앞 (무작위 분할 아님),통과,
8,학습 · 검증에 2025 데이터 없음,통과,
9,이력 피처가 미래 기록 없이도 똑같음,통과,"표본 20건 × 15개, 불일치 0개"


**이 노트북이 확인하지 못하는 것**: 코드가 의도대로 돌았는지는 확인하지만, 판단 자체
(비용을 100배 오입력으로 본 것, 반복 고장 드론을 지우지 않은 것 등)가 옳은지는 확인하지 못한다.
그런 판단의 근거는 `outputs/report.md` 1장에 적었다.